# Session 07 — Demo C: Smart CCTV Analytics (Revised 2026)
**Google Colab | YOLO + ByteTrack | Vertical line counting | Clear dashboard | Manual evaluation**

This version keeps the working YOLO detector and ByteTrack tracking approach, but defaults to a **vertical line** for the OpenCV pedestrian sample (`vtest.avi`). It counts directional **crossing events**, not unique persons or guaranteed occupancy. All cells can run sequentially in Colab.

**Steps:** Install → load sample or upload → configure → process video → plots → evaluate manually.

## Step 1 — Install YOLO
Requires an internet connection for the package and model weights. Choose **Runtime → Change runtime type → T4 GPU** if available; CPU also works, more slowly.

In [ ]:
%pip -q install ultralytics

## Step 2 — Load pedestrian sample or upload your own video
**Option A (default):** download the public OpenCV `vtest.avi` and convert the first 15 seconds to MP4. This needs internet. **Option B:** set `VIDEO_SOURCE='upload'` and choose an MP4/AVI/MOV from your computer. A download failure produces a clear error message; the other option still works.

The OpenCV sample is useful for a *pedestrian detection and line-crossing lab*, although it is not an actual entrance camera. Its crossing events should not be interpreted as real building arrivals/departures.

In [ ]:
import os, cv2, subprocess, numpy as np, requests
from pathlib import Path
from IPython.display import Video, display
from google.colab import files

VIDEO_SOURCE = 'sample'  # @param ['sample', 'upload']

if VIDEO_SOURCE == 'sample':
    url = 'https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/vtest.avi'
    avi_path = 'opencv_vtest.avi'
    input_video = 'cctv_sample.mp4'
    try:
        with requests.get(url, timeout=90, stream=True) as resp:
            resp.raise_for_status()
            with open(avi_path, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=1024*1024):
                    if chunk: f.write(chunk)
        subprocess.run(['ffmpeg','-y','-loglevel','error','-i',avi_path,'-t','15',
                        '-c:v','libx264','-pix_fmt','yuv420p',input_video],check=True)
    except Exception as exc:
        raise RuntimeError('Could not download the OpenCV sample. Change VIDEO_SOURCE to upload, or check Colab internet access.') from exc
else:
    uploaded = files.upload()
    if not uploaded: raise RuntimeError('No file selected. Please rerun Step 2 and select a video.')
    input_video = next(iter(uploaded))

cap = cv2.VideoCapture(input_video)
ok, sample = cap.read()
if not ok:
    cap.release()
    raise RuntimeError(f'Cannot decode {input_video}. Try another MP4/AVI/MOV file.')
source_fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
source_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
original_h, original_w = sample.shape[:2]
cap.release()
print(f'Ready: {input_video} | {original_w}x{original_h} | {source_fps:.1f} FPS | {source_frames} frames')
display(Video(input_video,embed=True,width=720))

## Step 3 — Choose the crossing line and clean overlays
For the OpenCV pedestrian sample, begin with a **vertical line at 50% of width**. In a left-to-right crossing, the event is called *Enter*. Right-to-left is called *Exit*. These are **arbitrary directional labels** for this sample, not actual building entry or exit.

A **crossing** requires a tracked person's center to be observed on opposite sides of the virtual line; near-line jitter is ignored with a margin. To reduce duplicate events, the program has a cooldown per tracking ID, but ID switches can still cause false or missed counts.

In [ ]:
# @title CCTV Demo Settings
MODEL_NAME = 'yolov8s.pt'                    # @param {type:'string'}
CONFIDENCE = 0.22                            # @param {type:'number'}
IMAGE_SIZE = 640                             # @param {type:'integer'}
MAX_FRAMES = 450                             # @param {type:'integer'}
LINE_ORIENTATION = 'vertical'                 # @param ['vertical','horizontal']
LINE_POSITION = 0.50                         # @param {type:'slider', min:0.15, max:0.85, step:0.05}
ENTRY_DIRECTION = 'negative_to_positive'     # @param ['negative_to_positive','positive_to_negative']
CROSSING_MARGIN = 0.02                       # @param {type:'slider', min:0.0, max:0.08, step:0.01}
CROSSING_COOLDOWN_FRAMES = 20                # @param {type:'integer'}
SHOW_BOXES = True                            # @param {type:'boolean'}
SHOW_IDS = True                              # @param {type:'boolean'}
SHOW_CONFIDENCE = False                      # @param {type:'boolean'}
SHOW_DASHBOARD = True                        # @param {type:'boolean'}
MAX_OUTPUT_WIDTH = 960                       # @param {type:'integer'}

assert 0.0 < LINE_POSITION < 1.0
assert 0 <= CROSSING_MARGIN < .15
assert MAX_FRAMES > 0 and CROSSING_COOLDOWN_FRAMES >= 0
assert LINE_ORIENTATION in ('vertical','horizontal')
assert ENTRY_DIRECTION in ('negative_to_positive','positive_to_negative')
print(f'Counting line: {LINE_ORIENTATION} at {LINE_POSITION:.0%} of frame; entry direction: {ENTRY_DIRECTION}')

## Step 4 — YOLO + ByteTrack: detect and count crossings
Each person's **center point** is tracked relative to the virtual line. A valid event requires confirmed opposite-side observations and the cooldown interval. **Visible now** is the number of YOLO detections in the current frame; **Entered / Exited** are cumulative crossing events up to that frame. The last video frame should match the summary and plots.

The notebook limits frame processing to `MAX_FRAMES`. Increasing `IMAGE_SIZE` may detect smaller people but uses more compute. Tracking IDs are not validated unique-person IDs.

**Tracker reset:** Each run creates a new `YOLO(MODEL_NAME)` object so tracker IDs start from a fresh tracking session. IDs may still skip numbers because tracks that were created but not displayed can consume IDs; the largest ID is not the number of people. Each validated crossing event is saved to `cctv_crossing_events.csv`.

In [ ]:
from ultralytics import YOLO
import time
import csv
from datetime import timedelta

class CrossingCounter:
    """Directional virtual-line counter per track ID with hysteresis and cooldown."""
    def __init__(self, entry_direction='negative_to_positive', cooldown=20):
        self.entry_direction=entry_direction
        self.cooldown=cooldown
        self.last_confirmed_side={}
        self.last_event_frame={}
        self.entries=0
        self.exits=0
        self.events=[]
    def observe(self, track_id, side, frame_index):
        if side == 0:
            return None
        previous=self.last_confirmed_side.get(track_id)
        if previous in (-1,1) and previous != side:
            last=self.last_event_frame.get(track_id,-10**9)
            if frame_index-last >= self.cooldown:
                direction='negative_to_positive' if previous == -1 else 'positive_to_negative'
                event='entry' if direction == self.entry_direction else 'exit'
                if event == 'entry': self.entries += 1
                else: self.exits += 1
                self.events.append((frame_index,track_id,event))
                event_record={'frame':frame_index+1,'track_id':track_id,'event':event}
                self.last_event_frame[track_id]=frame_index
                self.last_confirmed_side[track_id]=side
                return event_record
        self.last_confirmed_side[track_id]=side
        return None

# Always create a fresh model instance and tracker state for this video run.
# Do NOT reuse a YOLO object or a global tracker from a prior video.
model=YOLO(MODEL_NAME)
cap=cv2.VideoCapture(input_video)
if not cap.isOpened(): raise RuntimeError(f'Cannot open {input_video}')
fps=cap.get(cv2.CAP_PROP_FPS) or 24.0
raw_w=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
raw_h=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
scale=min(1.0, MAX_OUTPUT_WIDTH/max(raw_w,1))
W=max(2,int(raw_w*scale)//2*2)
H=max(2,int(raw_h*scale)//2*2)

raw_output='cctv_annotated_raw.mp4'
final_output='cctv_annotated_result.mp4'
writer=cv2.VideoWriter(raw_output,cv2.VideoWriter_fourcc(*'mp4v'),fps,(W,H))
if not writer.isOpened():
    cap.release()
    raise RuntimeError('Cannot create video writer')
axis=0 if LINE_ORIENTATION=='vertical' else 1
size_axis=W if axis==0 else H
line_at=int(size_axis*LINE_POSITION)
margin_px=max(3,int(size_axis*CROSSING_MARGIN))
counter=CrossingCounter(ENTRY_DIRECTION,CROSSING_COOLDOWN_FRAMES)
frame_counts=[]
entries_timeline=[]
exits_timeline=[]
all_ids=set()
crossing_log=[]
maximum=processed=0

try:
    while processed<MAX_FRAMES:
        ok,frame=cap.read()
        if not ok: break
        if (frame.shape[1],frame.shape[0])!=(W,H):
            frame=cv2.resize(frame,(W,H))
        prediction=model.track(frame,persist=True,tracker='bytetrack.yaml',classes=[0],
                               conf=CONFIDENCE,imgsz=IMAGE_SIZE,verbose=False)[0]
        boxes=prediction.boxes
        people_now=len(boxes) if boxes is not None else 0
        maximum=max(maximum,people_now)
        frame_counts.append(people_now)
        if boxes is not None and boxes.id is not None:
            coords=boxes.xyxy.cpu().numpy().astype(int)
            ids=boxes.id.int().cpu().tolist()
            confs=boxes.conf.cpu().tolist()
            for (x1,y1,x2,y2),track_id,confidence in zip(coords,ids,confs):
                track_id=int(track_id)
                all_ids.add(track_id)
                cx,cy=(x1+x2)//2,(y1+y2)//2
                distance=(cx if axis==0 else cy)-line_at
                side=-1 if distance < -margin_px else (1 if distance > margin_px else 0)
                event_record=counter.observe(track_id,side,processed)
                if event_record is not None:
                    event_record.update({
                        'timestamp_s': round(processed / fps, 3),
                        'direction': ('left_to_right' if axis==0 else 'top_to_bottom')
                                     if side==1 else ('right_to_left' if axis==0 else 'bottom_to_top'),
                        'confidence': round(float(confidence), 3),
                        'center_x': cx, 'center_y': cy
                    })
                    crossing_log.append(event_record)
                if SHOW_BOXES:
                    cv2.rectangle(frame,(x1,y1),(x2,y2),(60,220,80),2)
                if SHOW_IDS or SHOW_CONFIDENCE:
                    label=('ID '+str(track_id) if SHOW_IDS else '')
                    if SHOW_CONFIDENCE: label += (' ' if label else '')+f'{confidence:.2f}'
                    font_scale=max(.36,min(.54,W/1600))
                    (tw,th),baseline=cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,font_scale,1)
                    lx=max(0,min(x1,W-tw-8))
                    ly=max(th+8,y1-4)
                    cv2.rectangle(frame,(lx,ly-th-5),(lx+tw+7,ly+baseline+2),(25,65,31),-1)
                    cv2.putText(frame,label,(lx+3,ly),cv2.FONT_HERSHEY_SIMPLEX,font_scale,
                                (255,255,255),1,cv2.LINE_AA)
        if LINE_ORIENTATION=='vertical':
            cv2.line(frame,(line_at,0),(line_at,H),(20,235,240),2)
        else:
            cv2.line(frame,(0,line_at),(W,line_at),(20,235,240),2)
        if SHOW_DASHBOARD:
            overlay=frame.copy()
            panel_w=min(W,500)
            panel_h=min(H,106)
            cv2.rectangle(overlay,(0,0),(panel_w,panel_h),(14,25,28),-1)
            cv2.addWeighted(overlay,.82,frame,.18,0,frame)
            fs=.60 if W>=600 else .42
            gap=29 if panel_h>=100 else 18
            cv2.putText(frame,'SMART CCTV | YOLO + ByteTrack',(12,22),cv2.FONT_HERSHEY_SIMPLEX,
                        fs,(255,255,255),2,cv2.LINE_AA)
            cv2.putText(frame,f'Visible now: {people_now} | Max: {maximum}',(12,22+gap),
                        cv2.FONT_HERSHEY_SIMPLEX,fs,(105,240,125),2,cv2.LINE_AA)
            cv2.putText(frame,f'Crossings IN: {counter.entries} | OUT: {counter.exits}',(12,22+2*gap),
                        cv2.FONT_HERSHEY_SIMPLEX,fs,(80,230,245),2,cv2.LINE_AA)
        writer.write(frame)
        entries_timeline.append(counter.entries)
        exits_timeline.append(counter.exits)
        processed+=1
        if processed % 60==0: print(f'Processed {processed} frames ...')
finally:
    cap.release()
    writer.release()

if processed==0: raise RuntimeError('No frames processed')
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',raw_output,'-c:v','libx264',
                '-pix_fmt','yuv420p','-movflags','+faststart',final_output],check=True)
print(f'Processed frames: {processed}')
print(f'Maximum detections/frame: {maximum}; average: {np.mean(frame_counts):.1f}')
print(f'Crossings IN: {counter.entries} | OUT: {counter.exits} | total events: {len(counter.events)}')
print(f'Tracking IDs observed: {len(all_ids)} (NOT verified unique people)')
# Export an audit trail of crossing EVENTS (not a unique-person census).
log_file='cctv_crossing_events.csv'
log_columns=['frame','timestamp_s','track_id','event','direction','confidence','center_x','center_y']
with open(log_file,'w',newline='',encoding='utf-8') as fh:
    csv_writer=csv.DictWriter(fh,fieldnames=log_columns)
    csv_writer.writeheader()
    csv_writer.writerows(crossing_log)
assert len(crossing_log) == counter.entries + counter.exits
print('Event log:',log_file,'| records:',len(crossing_log))
print('Output:',final_output)
display(Video(final_output,embed=True,width=800))

## Step 5 — Detection and crossing-event charts
The first chart shows detections per frame. The second shows **cumulative virtual-line crossing events**. Crossing counts are not a reliable estimate of real-world occupancy without a properly defined entrance, initial occupancy, and validated tracks.

The next section also prints a CSV event log: **frame, timestamp (seconds), tracking ID, entry/exit, motion direction, detection confidence, and bounding-box center**. Review the actual frame before judging whether each event is correct.

In [ ]:
import matplotlib.pyplot as plt
frames_axis=np.arange(1,len(frame_counts)+1)
plt.figure(figsize=(11,3.5))
plt.plot(frames_axis,frame_counts,linewidth=1.6,label='People detected per frame')
plt.axhline(np.mean(frame_counts),linestyle='--',label=f'Average {np.mean(frame_counts):.1f}')
plt.title('Smart CCTV — Detection Per Frame')
plt.xlabel('Frame');plt.ylabel('Number of detections')
plt.grid(alpha=.2);plt.legend();plt.tight_layout();plt.show()

plt.figure(figsize=(11,3.5))
plt.step(frames_axis,entries_timeline,where='post',label='Crossing IN')
plt.step(frames_axis,exits_timeline,where='post',label='Crossing OUT')
plt.title('Smart CCTV — Cumulative Virtual Line Crossings')
plt.xlabel('Frame');plt.ylabel('Crossing events')
plt.grid(alpha=.2);plt.legend();plt.tight_layout();plt.show()
print('Final crossings: IN =',counter.entries,'OUT =',counter.exits)
print('To download the annotated video, run: files.download(final_output)')

## Step 6 — Audit crossing events (CSV)
This table shows **crossing events**, not distinct people. The video and CSV are downloadable for manual verification. If the table is empty, the CSV will contain only its header.

**Classroom check:** pick each event frame, pause the annotated MP4 near that timestamp, and mark it true / false. A single track may legitimately trigger multiple crossings if it reverses direction.

In [ ]:
import pandas as pd
from IPython.display import display
events_df = pd.read_csv(log_file)
if events_df.empty:
    print('No virtual-line crossing events were logged.')
else:
    display(events_df)
print(f'Event records: {len(events_df)} | IN: {counter.entries} | OUT: {counter.exits}')
# To download both: uncomment next lines after processing.
# files.download(log_file)
# files.download(final_output)

## Step 7 — Manual checking (10-minute student activity)
**Pause the annotated video** at several positions near the virtual line, or replay the raw video slowly. Record **actual crossing events** for the full clip by direction, not merely the number of people visible. Enter those two counts below to calculate absolute count error.

This is a simple aggregate counting check. For a stronger evaluation, students should annotate the time and direction of each manual crossing, match events with predictions, then calculate precision/recall, false crossings, and missed crossings.

**Discussion:** Why can a tracked person who disappears and reappears receive a new ID? Why can an ID switch introduce a false crossing? How would you verify which side of the line should be called *entry*?

In [ ]:
# @title Manual crossing count evaluation (fill in after viewing the full clip)
MANUAL_IN = -1    # @param {type:'integer'}
MANUAL_OUT = -1   # @param {type:'integer'}
if min(MANUAL_IN,MANUAL_OUT)<0:
    print('Set MANUAL_IN and MANUAL_OUT to your observed counts (0 is allowed).')
else:
    in_error=abs(counter.entries-MANUAL_IN)
    out_error=abs(counter.exits-MANUAL_OUT)
    print(f'IN:  manual={MANUAL_IN}, YOLO crossings={counter.entries}, absolute error={in_error}')
    print(f'OUT: manual={MANUAL_OUT}, YOLO crossings={counter.exits}, absolute error={out_error}')
    print(f'Total absolute directional count error: {in_error+out_error}')
    print('Note: this is an aggregate count error, not precision/recall or a unique-person metric.')